# A movie catalog agent: MongoDB queries + Brave Search, no embeddings

An agent over the `sample_mflix.movies` collection that Atlas loads with its
sample data (about 21,000 movies up to 2016). Where
[`mongodb_text_search.ipynb`](mongodb_text_search.ipynb) retrieves passages of
a PDF, this one queries structured documents: it filters by director, cast,
genre, year and rating with regular `find` queries, and searches plots with
MongoDB's **native text index** (`$text`), which the sample data already
includes. No embeddings, and no Atlas Search index either.

**Brave Search** adds what a 2016 snapshot can't have: where a movie streams
today, what happened to it since, sequels, box office. The rule the agent
follows is that **the catalog decides what exists**: before any web search it
checks that the movie is in the collection, and if it isn't, it says so and
doesn't look it up on the web.

Needs `MONGODB_CONNECTION` (an Atlas cluster with the sample dataset loaded)
and `BRAVE_API_KEY` in `.env`.

In [1]:
import json
import os
import re
import threading
import time

import httpx
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pymongo import DESCENDING, MongoClient
from strands import Agent, tool
from strands.models import BedrockModel

load_dotenv()
YOUR_BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-5-5"

mongo_client = MongoClient(os.environ["MONGODB_CONNECTION"])
movies = mongo_client["sample_mflix"]["movies"]
print(f"{movies.estimated_document_count():,} movies in sample_mflix.movies")

brave_api_key = os.environ.get("BRAVE_API_KEY", "")
print(f"Brave API key available: {'Yes' if brave_api_key else 'No'}")

21,349 movies in sample_mflix.movies
Brave API key available: Yes


## 1. The data

One document per movie. The fields the tools use:

In [2]:
FIELDS = {
    "_id": 0, "title": 1, "year": 1, "genres": 1, "directors": 1, "cast": 1,
    "plot": 1, "fullplot": 1, "runtime": 1, "released": 1, "countries": 1,
    "imdb.rating": 1, "imdb.votes": 1, "awards.text": 1,
    "tomatoes.critic.meter": 1, "tomatoes.viewer.meter": 1,
    "tomatoes.consensus": 1, "tomatoes.boxOffice": 1,
}

print(json.dumps(movies.find_one({"title": "Inception"}, FIELDS), indent=1, default=str))

{
 "fullplot": "Dom Cobb is a skilled thief, the absolute best in the dangerous art of extraction, stealing valuable secrets from deep within the subconscious during the dream state, when the mind is at its most vulnerable. Cobb's rare ability has made him a coveted player in this treacherous new world of corporate espionage, but it has also made him an international fugitive and cost him everything he has ever loved. Now Cobb is being offered a chance at redemption. One last job could give him his life back but only if he can accomplish the impossible-inception. Instead of the perfect heist, Cobb and his team of specialists have to pull off the reverse: their task is not to steal an idea but to plant one. If they succeed, it could be the perfect crime. But no amount of careful planning or expertise can prepare the team for the dangerous enemy that seems to predict their every move. An enemy that only Cobb could have seen coming.",
 "imdb": {
  "rating": 8.8,
  "votes": 1294646
 },
 "y

`$text` needs a text index, and a collection can only have one. The sample
data ships with it; the check below creates it only if it's missing.

Two quirks of the sample data the tools work around: 35 movies have `year` as
a string (`"1981è"`) and 61 have an empty string for `imdb.rating`. Filtering
on `$type: "number"` keeps them out of range filters and rating sorts.

In [3]:
has_text_index = any(
    "text" in index["key"].values() for index in movies.list_indexes()
)
if not has_text_index:
    movies.create_index(
        [("title", "text"), ("fullplot", "text"), ("cast", "text"), ("genres", "text")],
        name="movies_text",
    )
print("Text index:", [i["name"] for i in movies.list_indexes() if "text" in i["key"].values()])

Text index: ['cast_text_fullplot_text_genres_text_title_text']


## 2. The tools

Three read MongoDB and one reads the web:

- `find_movies` builds a `find` filter from structured arguments. The model
  fills in the arguments; it never writes a query.
- `search_plots` is the keyword search over the text index: "a heist that
  goes wrong" matches plots by their words, stemmed.
- `movie_details` returns everything the catalog knows about one title. It's
  the gate for the web: the agent calls it before `web_search`, and a miss
  means the movie isn't in the catalog.
- `web_search` calls the Brave Search API directly with `httpx`.

In [4]:
# Movies with a numeric rating (61 have an empty string instead). Needed for any
# sort by rating: in BSON order a string sorts above every number.
HAS_RATING = {"imdb.rating": {"$type": "number"}}


def summarize(doc):
    """One line per movie, as the tools return them."""
    rating = doc.get("imdb", {}).get("rating")
    rating = f"{rating}" if isinstance(rating, (int, float)) else "n/a"
    return (
        f"- {doc.get('title')} ({doc.get('year')}) — IMDb {rating}; "
        f"{', '.join(doc.get('genres', []))}; "
        f"dir. {', '.join(doc.get('directors', [])) or 'n/a'}; "
        f"cast: {', '.join(doc.get('cast', [])[:4]) or 'n/a'}"
    )


@tool
def find_movies(
    person: str | None = None,
    genre: str | None = None,
    year_from: int | None = None,
    year_to: int | None = None,
    min_rating: float | None = None,
    limit: int = 10,
) -> str:
    """List movies in the catalog that match every given filter, best rated first.

    Args:
        person: Full name of a director or cast member, e.g. "Christopher Nolan".
        genre: One genre, e.g. "Sci-Fi", "Drama", "Film-Noir".
        year_from: Earliest release year, inclusive.
        year_to: Latest release year, inclusive.
        min_rating: Minimum IMDb rating, 0-10.
        limit: Maximum number of movies to return.
    """
    query = dict(HAS_RATING)
    if person:
        # Exact name, case-insensitive: a regex would also match "Tom Hanks Jr."
        name = re.compile(f"^{re.escape(person)}$", re.IGNORECASE)
        query["$or"] = [{"directors": name}, {"cast": name}]
    if genre:
        query["genres"] = re.compile(f"^{re.escape(genre)}$", re.IGNORECASE)
    if year_from or year_to:
        query["year"] = {"$type": "number"}
        if year_from:
            query["year"]["$gte"] = year_from
        if year_to:
            query["year"]["$lte"] = year_to
    if min_rating is not None:
        query["imdb.rating"]["$gte"] = min_rating

    total = movies.count_documents(query)
    docs = list(movies.find(query, FIELDS).sort("imdb.rating", DESCENDING).limit(limit))
    print(f"[find_movies] {json.dumps({k: v for k, v in locals().items() if k in ('person', 'genre', 'year_from', 'year_to', 'min_rating') and v})}: {total} matches")

    if not docs:
        return "No movies in the catalog match these filters."
    return f"{total} matches, showing {len(docs)}:\n" + "\n".join(summarize(d) for d in docs)


@tool
def search_plots(query: str, limit: int = 8) -> str:
    """Keyword search over titles, plots, cast and genres in the catalog.

    Lexical, not semantic: it matches the words of the query (stemmed) against
    the text of each movie. Use English keywords, e.g. "bank heist hostages".

    Args:
        query: English keywords describing the movie or its plot.
        limit: Maximum number of movies to return.
    """
    docs = list(
        movies.find({"$text": {"$search": query}}, {**FIELDS, "score": {"$meta": "textScore"}})
        .sort([("score", {"$meta": "textScore"})])
        .limit(limit)
    )
    print(f"[search_plots] {query!r}: {len(docs)} matches")

    if not docs:
        return "No movies in the catalog match these keywords."
    return "\n".join(
        f"{summarize(d)}\n  {d.get('plot', '')}" for d in docs
    )


@tool
def movie_details(title: str, year: int | None = None) -> str:
    """Everything the catalog has on one movie. Call it before any web search:
    a movie that isn't found here is not in the catalog.

    Args:
        title: Exact title, case-insensitive.
        year: Release year, to tell remakes apart.
    """
    query = {"title": re.compile(f"^{re.escape(title)}$", re.IGNORECASE)}
    if year:
        query["year"] = year
    docs = list(movies.find(query, FIELDS))
    print(f"[movie_details] {title!r}{f' ({year})' if year else ''}: {len(docs)} found")

    if not docs:
        return f"NOT IN CATALOG: no movie titled {title!r}" + (f" from {year}" if year else "") + "."
    if len(docs) > 1:
        return "Several movies share this title, call again with a year:\n" + "\n".join(summarize(d) for d in docs)
    return json.dumps(docs[0], indent=1, default=str)


BRAVE_SEARCH_URL = "https://api.search.brave.com/res/v1/web/search"

# Brave's free plan allows one request per second, and Strands runs the tools of
# a turn concurrently: the lock spaces the requests out instead of getting 429s.
BRAVE_MIN_INTERVAL = 1.1
brave_lock = threading.Lock()
brave_last_request = 0.0

# Brave highlights the matched words in the snippets with <strong>.
HTML_TAG = re.compile(r"<[^>]+>")


@tool
def web_search(query: str, count: int = 5) -> str:
    """Search the web with Brave Search for more about a movie that IS in the catalog:
    where to stream it now, news since its release, sequels, box office, trivia.

    Never use it for a movie that movie_details did not find.

    Args:
        query: The search query, including the title and year of the movie.
        count: Number of results to return (at most 20).
    """
    global brave_last_request

    with brave_lock:
        wait = BRAVE_MIN_INTERVAL - (time.monotonic() - brave_last_request)
        if wait > 0:
            time.sleep(wait)
        response = httpx.get(
            BRAVE_SEARCH_URL,
            params={"q": query, "count": min(count, 20)},
            headers={
                "Accept": "application/json",
                "X-Subscription-Token": brave_api_key,
            },
            timeout=15,
        )
        brave_last_request = time.monotonic()

    response.raise_for_status()
    results = response.json().get("web", {}).get("results", [])
    print(f"[web_search] {query!r}: {len(results)} results")

    if not results:
        return "No web results found."

    return "\n\n".join(
        f"{result['title']}\n{result['url']}\n"
        f"{HTML_TAG.sub('', result.get('description', ''))}"
        for result in results
    )

Tools decorated with `@tool` can still be called as plain functions:

In [5]:
print(find_movies(person="Christopher Nolan", min_rating=8))

[find_movies] {"person": "Christopher Nolan", "min_rating": 8}: 7 matches
7 matches, showing 7:
- The Dark Knight (2008) — IMDb 9.0; Action, Crime, Drama; dir. Christopher Nolan; cast: Christian Bale, Heath Ledger, Aaron Eckhart, Michael Caine
- Inception (2010) — IMDb 8.8; Action, Mystery, Sci-Fi; dir. Christopher Nolan; cast: Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy
- Interstellar (2014) — IMDb 8.7; Adventure, Drama, Sci-Fi; dir. Christopher Nolan; cast: Ellen Burstyn, Matthew McConaughey, Mackenzie Foy, John Lithgow
- Memento (2000) — IMDb 8.5; Mystery, Thriller; dir. Christopher Nolan; cast: Guy Pearce, Carrie-Anne Moss, Joe Pantoliano, Mark Boone Junior
- The Prestige (2006) — IMDb 8.5; Drama, Mystery, Thriller; dir. Christopher Nolan; cast: Hugh Jackman, Christian Bale, Michael Caine, Piper Perabo
- The Dark Knight Rises (2012) — IMDb 8.5; Action, Thriller; dir. Christopher Nolan; cast: Christian Bale, Gary Oldman, Tom Hardy, Joseph Gordon-Levitt
- Batman Be

In [6]:
print(search_plots("bank heist hostages", limit=3))

[search_plots] 'bank heist hostages': 3 matches
- Federal Bank Heist (2011) — IMDb 5.8; Crime, Drama, Thriller; dir. Marcos Paulo; cast: Milhem Cortaz, Eriberto Leèo, Hermila Guedes, Lima Duarte
  "The Baron" wanted to commit the perfect heist involving 3 tons of money and no violence. For this he would need the right people willing to get 1 million dollars to take part in this job. ...
- The Bank (2001) — IMDb 6.7; Thriller, Drama; dir. Robert Connolly; cast: David Wenham, Anthony LaPaglia, Sibylla Budd, Steve Rodgers
  The Bank is a thriller about banking, corruption and alchemy.
- Children of Hannibal (1998) — IMDb 6.3; Comedy; dir. Davide Ferrario; cast: Diego Abatantuono, Silvio Orlando, Valentina Cervi, Flavio Insinna
  A bungling bank robber, an uncooperative hostage with a secret, a gay policeman and a situation which rapidly goes from bad to worse are the ingredients to this very funny and unpredictable...


## 3. The agent

The system prompt spells out the catalog-first rule. Without it the model
happily answers about any movie from the web, which is exactly what this agent
must not do.

In [7]:
SYSTEM_PROMPT = """You are an assistant for a movie catalog stored in MongoDB (movies up to 2016), with web search to add current information about movies in that catalog.

## Rules

1. The catalog decides which movies exist. Before any web search about a movie, call `movie_details` to confirm it is in the catalog.
2. If `movie_details` returns NOT IN CATALOG, tell the user the movie is not in the catalog and stop. Do not search the web for it and do not describe it from memory.
3. Use `web_search` only to add to what the catalog says about a movie that IS in it: where to stream it now, news since release, sequels and remakes, box office, trivia, current critical standing. Include the title and year in the query.
4. Facts about plots, cast, crew, ratings and awards come from the catalog. Say what comes from the catalog and what comes from the web, with the URL for web facts.
5. Answer in the language of the question. Keep answers concise.
"""

model = BedrockModel(
    model_id=YOUR_BEDROCK_MODEL_ID,
    region_name=os.getenv("AWS_REGION", "us-west-2"),
    max_tokens=4096,
)


def ask(question):
    # A fresh agent per question, so earlier answers don't leak into the next
    agent = Agent(
        name="Movie Catalog Agent",
        model=model,
        system_prompt=SYSTEM_PROMPT,
        tools=[find_movies, search_plots, movie_details, web_search],
        callback_handler=None,  # don't also stream the answer to stdout
    )

    response = agent(question)

    # Tools the agent actually called (absent from the dict means never called)
    used = response.metrics.tool_metrics
    usage = response.metrics.accumulated_usage
    display(Markdown(
        "**Tools used:** "
        + (", ".join(f"`{name}` x{m.call_count}" for name, m in used.items()) or "_none_")
        + f" — {usage['inputTokens']:,} input / {usage['outputTokens']:,} output tokens"
    ))

    display(Markdown(str(response)))

A movie in the catalog: the plot, cast and ratings come from MongoDB, the
streaming options from Brave.

In [8]:
ask("Tell me about Inception and where I can stream it today.")

[movie_details] 'Inception': 1 found
[web_search] 'Inception 2010 where to stream': 5 results


**Tools used:** `movie_details` x1, `web_search` x1 — 6,713 input / 714 output tokens

**Inception (2010)** (from the catalog)
- **Director:** Christopher Nolan
- **Cast:** Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy
- **Genres:** Action, Mystery, Sci-Fi
- **Runtime:** 148 min
- **Plot:** Dom Cobb is a thief who steals corporate secrets by entering people's dreams. He gets one last job that could win him his old life back. The job is "inception", which means planting an idea in a CEO's mind instead of stealing one.
- **IMDb rating:** 8.8 (about 1.29M votes)
- **Rotten Tomatoes:** critics 86%, viewers 91%
- **Awards:** Won 4 Oscars, plus 158 other wins and 172 nominations.
- **Box office:** $292.6M

**Where to stream (from the web)**
The sources I found disagree, and I can't tell which is current:
- [JustWatch](https://www.justwatch.com/us/movie/inception) lists Netflix, Prime Video and Disney+.
- [IMDb](https://www.imdb.com/news/ni63927433/) says HBO Max and Netflix.
- [Reelgood](https://reelgood.com/us/movie/inception-2010) says ViX (via Roku).
- [Yahoo](https://www.yahoo.com/entertainment/inception-2010-streaming-watch-stream-035946523.html) says Hulu.
- [Prime Video](https://www.amazon.com/Inception-Leonardo-DiCaprio/dp/B0047WJ11G) has a page for it, probably to rent or buy.

Streaming lineups change often and differ by country. Check the JustWatch page for your region to see what's available today.


Catalog query first, then the web for just the one movie it picked:

In [9]:
ask("¿Qué películas de Christopher Nolan hay en el catálogo con nota IMDb mayor a 8? "
    "De la mejor valorada, ¿qué dice la crítica hoy y tuvo alguna secuela?")

[find_movies] {"person": "Christopher Nolan", "min_rating": 8.0}: 7 matches
[movie_details] 'The Dark Knight' (2008): 1 found
[web_search] 'The Dark Knight (2008) critical reception today Rotten Tomatoes legacy sequel The Dark Knight Rises': 5 results


**Tools used:** `find_movies` x1, `movie_details` x1, `web_search` x1 — 10,288 input / 894 output tokens

**Películas de Christopher Nolan con IMDb mayor a 8 (catálogo):**
1. The Dark Knight (2008): 9.0
2. Inception (2010): 8.8
3. Interstellar (2014): 8.7
4. Memento (2000): 8.5
5. The Prestige (2006): 8.5
6. The Dark Knight Rises (2012): 8.5
7. Batman Begins (2005): 8.3

**La mejor valorada es The Dark Knight (9.0).**

**Crítica (catálogo):** Tiene un 94% de críticos y un 94% de público en Rotten Tomatoes. El consenso dice: "Dark, complex and unforgettable, The Dark Knight succeeds not just as an entertaining comic book film, but as a richly thrilling crime saga." Ganó 2 Oscars y, según el catálogo, 142 premios más. Recaudó $533,3M.

**Crítica hoy (web):** La ficha de Rotten Tomatoes mantiene el mismo consenso, lo que indica que la valoración sigue siendo muy positiva. Los resultados de la búsqueda no mostraron datos más recientes. Fuente: https://www.rottentomatoes.com/m/the_dark_knight

**¿Secuela?** Sí, *The Dark Knight Rises* (2012), que cierra la trilogía de Nolan. Está en el catálogo con IMDb 8.5. Según Rotten Tomatoes (web), tiene un 87% de aprobación y la describen como una película "ambitious, thoughtful, and potent" que concluye la franquicia de Nolan "en forma espectacular". Fuentes: https://www.rottentomatoes.com/m/the_dark_knight_rises y https://greatestmovies.fandom.com/wiki/The_Dark_Knight_Rises


A description instead of a title goes through the text index:

In [10]:
ask("I'm looking for a 1990s movie about a bank heist that goes wrong, rated above 7. "
    "What is it, and is a remake in the works?")

[find_movies] {"genre": "Crime", "year_from": 1990, "year_to": 1999, "min_rating": 7.0}: 189 matches
[search_plots] 'bank heist goes wrong': 15 matches
[movie_details] 'Heat' (1995): 1 found
[search_plots] 'bank robbery robbers hostages botched police': 10 matches
[web_search] 'Heat (1995) Michael Mann remake OR sequel Heat 2 news': 5 results


**Tools used:** `find_movies` x1, `search_plots` x2, `movie_details` x1, `web_search` x1 — 26,499 input / 1,264 output tokens

**The movie:** *Heat* (1995) is my best match. The catalog gives these details:
- It was directed by Michael Mann and stars Al Pacino, Robert De Niro, Val Kilmer and Jon Voight.
- Its IMDb rating is 8.3.
- The plot: a crew of professional bank robbers comes under police pressure after a botched job leaves a clue behind. Lt. Vincent Hanna's team then hunts them.

I didn't find another 1990s bank-heist film above 7 in the catalog. Other 1990s crime films like *Reservoir Dogs* (1992) involve heists, but I didn't check whether they fit as well.

**Remake:** I found no remake, but a sequel is in the works. This comes from the web, not the catalog.
- *Heat 2* would be a film adaptation of the 2022 novel by Michael Mann and Meg Gardiner. The novel is both a prequel and a sequel to the 1995 film. Mann is set to write and direct. (https://en.wikipedia.org/wiki/Heat_(1995_film))
- The Hollywood Reporter says the project moved from Warner Bros. to United Artists, with Jerry Bruckheimer and Scott Stuber producing. Leonardo DiCaprio was reported as circling the lead. (https://www.hollywoodreporter.com/movies/movie-news/heat-2-is-on-michael-mann-crime-thriller-1236392878/)
- Wikipedia lists DiCaprio and Christian Bale as stars. (https://en.wikipedia.org/wiki/Heat_(1995_film))
- Deadline has an update from Mann on when he plans to start production. (https://deadline.com/2024/07/michael-mann-heat-2-filming-update-casting-rumors-1236012108/)

I didn't open these articles, so I can't confirm the current production status.


A movie released after the snapshot. It's not in the catalog, so the agent
must say so and skip the web, however well it knows the movie.

In [11]:
ask("Tell me about Oppenheimer (2023) and where to stream it.")

[movie_details] 'Oppenheimer' (2023): 0 found


**Tools used:** `movie_details` x1 — 3,090 input / 210 output tokens

I can't help with *Oppenheimer* (2023) because it isn't in the catalog. The catalog only covers movies up to 2016. Since the film isn't in it, I haven't searched the web for streaming details or described it from memory.

I can look for other movies in the catalog instead. For example, I could search by director (Christopher Nolan's earlier films), by genre, or by a topic like the Manhattan Project.


## Clean up

The notebook only reads the sample collection, so there's nothing to drop.

In [12]:
mongo_client.close()